In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()

In [2]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [ ]:
fecha_corte = pd.to_datetime("2026-02-28")

# Rango de análisis 
fecha_inicio_12m = fecha_corte - pd.DateOffset(months=12)
fecha_inicio_24m = fecha_corte - pd.DateOffset(months=24)

In [4]:
# ===============================
# Datos de clientes y ventas
# ===============================
df_clientes = pd.read_sql(f"""
SELECT Cliente, 
CosechaFecha 
FROM Ventas_Comerssia.dbo.Cliente_Perfil
WHERE CosechaFecha <= '{fecha_corte}'""", engine)
df_clientes["CosechaFecha"] = pd.to_datetime(df_clientes["CosechaFecha"], errors="coerce")

query_ventas = f"""
SELECT Cliente, Fecha, Venta_Neta AS Venta
FROM Ventas_Comerssia.dbo.Ventas_Unificadas
WHERE Fecha BETWEEN '{fecha_inicio_24m:%Y-%m-%d}' AND '{fecha_corte:%Y-%m-%d}'
"""
df_ventas = pd.read_sql(query_ventas, engine)
df_ventas["Fecha"] = pd.to_datetime(df_ventas["Fecha"], errors="coerce")

In [5]:
# ===============================
# 3) Métricas por cliente
# ===============================
# Última compra 
ultima_compra = (
    df_ventas.groupby("Cliente")["Fecha"]
    .max()
    .reset_index()
    .rename(columns={"Fecha": "UltimaCompra"})
)

# Ventas últimos 12 meses (para SegmentoActual)
ventas_12m = (
    df_ventas[df_ventas["Fecha"] >= fecha_inicio_12m]
    .groupby("Cliente")["Venta"]
    .sum()
    .reset_index()
    .rename(columns={"Venta": "Venta12M"})
)

# Ventas 24 meses (para Segmento24M)
ventas_24m = (
    df_ventas.groupby("Cliente")["Venta"]
    .sum()
    .reset_index()
    .rename(columns={"Venta": "Venta24M"})
)

# ===============================
# 4) Merge sobre la base completa de clientes
# ===============================
clientes = df_clientes.merge(ultima_compra, on="Cliente", how="left")
clientes = clientes.merge(ventas_12m, on="Cliente", how="left")
clientes = clientes.merge(ventas_24m, on="Cliente", how="left")


In [6]:
# ===============================
# 5) Función exacta para asignar segmento por monto 
# ===============================
def segmento_por_valor(valor):
    # Nota: 0 => "Sin Segmento"
    if pd.isna(valor) or valor == 0:
        return "Sin Segmento"
    if valor > 1_400_000:
        return "Diamante"
    if 700_000 <= valor <= 1_400_000:
        return "Oro"
    if 300_000 <= valor < 700_000:
        return "Plata"
    if 0 < valor < 300_000:
        return "Bronce"
    return "Sin Segmento"

clientes["Segmento12M"] = clientes["Venta12M"].apply(segmento_por_valor)
clientes["Segmento24M"] = clientes["Venta24M"].apply(segmento_por_valor)

# ===============================
# 6) Recencia en días 
# ===============================
# calcular dias desde ultima compra 
clientes["RecenciaDias"] = (fecha_corte - clientes["UltimaCompra"]).dt.days

def calcular_recencia(row):
    # Si no tiene ultima compra -> devolvemos 
    if pd.isna(row["UltimaCompra"]):
        return None

    # Nuevo: basado en cosecha (<= 90 dias)
    if pd.notna(row["CosechaFecha"]):
        diff_cosecha = (fecha_corte - row["CosechaFecha"]).days
        if diff_cosecha <= 90:
            return "Nuevo"

    d = row["RecenciaDias"]
    if d <= 120:
        return "Muy Activo"
    if 121 <= d <= 300:
        return "Activo"
    if 301 <= d <= 365:
        return "Por Inactivar"
    if 366 <= d <= 395:
        return "Churn"
    if d > 395:
        return "Inactivo"
    return None

clientes["Recencia"] = clientes.apply(calcular_recencia, axis=1)

# ===============================
# 7) Definir Segmento final 
# ===============================
def asignar_segmento_final(row):
    rec = row["Recencia"]
    if rec in ["Nuevo", "Muy Activo", "Activo", "Por Inactivar"]:
        return row["Segmento12M"]
    if rec in ["Churn", "Inactivo"]:
        return row["Segmento24M"]
    # Si no tiene recencia (no compró) -> si tiene ventas24M ==0 -> Sin Segmento
    return "Sin Segmento"

clientes["Segmento"] = clientes.apply(asignar_segmento_final, axis=1)

# ===============================
# 8) Regla final: si Segmento 
# ===============================
clientes.loc[clientes["Segmento"] == "Sin Segmento", "Recencia"] = None

In [7]:
# ===============================
# 9) Resultado final (columns)
# ===============================
resultado = clientes[[
    "Cliente", "CosechaFecha", "UltimaCompra",
    "Venta12M", "Venta24M",
    "Recencia", "Segmento"
]]

# Mostrar ejemplo
print(resultado.head(50))

        Cliente CosechaFecha UltimaCompra   Venta12M    Venta24M  \
0    C900099007   2021-12-12          NaT        NaN         NaN   
1   C1012335241   2025-12-20   2025-12-20   63025.21    63025.21   
2     C19102757   2021-04-06   2025-02-13        NaN   165756.30   
3     C63346697   2024-12-24   2024-12-24        NaN    66386.55   
4       C431666   2021-02-19          NaT        NaN         NaN   
5     C49688978   2023-12-11   2025-10-25  817470.29  2273298.81   
6   C1018514781   2024-04-15   2025-04-10  127731.09   248739.49   
7     C16751060   2020-06-12   2025-12-17   21008.40    95798.32   
8     C45421081   2025-12-18   2025-12-18   83193.28    83193.28   
9       C497199   2022-04-30          NaT        NaN         NaN   
10  C1016003196   2022-02-15          NaT        NaN         NaN   
11    C67036003   2021-09-20          NaT        NaN         NaN   
12  C1140818183   2024-12-20   2025-12-21   49159.66   248319.34   
13  C1136884060   2022-07-30          NaT       

In [8]:
resultado.to_excel("segmentacion_clientes.xlsx", index=False)

In [9]:
resultado.to_sql("Segmentacion_Clientes", engine, if_exists="replace", index=False)

272